# Notebook 01 — CNN+CTC sobre EMNIST

**Proyecto IA — Sistema OCR Educativo**

Este notebook recorre paso a paso el entrenamiento de una red **CRNN+CTC** sobre
EMNIST. El objetivo es comprender:

1. Cómo se prepara el dataset.
2. Qué hace cada capa del modelo.
3. Por qué necesitamos la pérdida CTC.
4. Cómo se decodifica la salida del modelo.

## 1. Importar librerías

In [ ]:
import sys, os
sys.path.append(os.path.abspath('../modelo_personalizado'))

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from modelo import construir_modelo_entrenamiento, construir_modelo_inferencia, ALFABETO
from entrenar import cargar_emnist, generar_pseudo_palabras
from decodificador_ctc import decodificar_greedy

print('TensorFlow:', tf.__version__)

## 2. Cargar EMNIST y visualizar muestras

In [ ]:
imgs, lbls = cargar_emnist()
print('Forma del conjunto:', imgs.shape)

fig, axes = plt.subplots(2, 8, figsize=(12, 3))
for ax, img, lbl in zip(axes.flat, imgs[:16], lbls[:16]):
    ax.imshow(img, cmap='gray')
    ax.set_title(ALFABETO[lbl] if lbl < len(ALFABETO) else '?')
    ax.axis('off')
plt.tight_layout(); plt.show()

## 3. Construir pseudo-palabras para entrenar con CTC

EMNIST tiene caracteres aislados. Para entrenar reconocimiento de **secuencias**
concatenamos varios caracteres en una imagen de 32×128 pixeles.

In [ ]:
X, Y = generar_pseudo_palabras(imgs, lbls, n_muestras=2000, long_max=6)
print('X:', X.shape, '   Y:', Y.shape)

fig, axes = plt.subplots(4, 1, figsize=(8, 6))
for ax, x, y in zip(axes, X[:4], Y[:4]):
    txt = ''.join(ALFABETO[i] for i in y if i >= 0)
    ax.imshow(x.squeeze(), cmap='gray'); ax.set_title(f'\"{txt}\"'); ax.axis('off')
plt.tight_layout(); plt.show()

## 4. Construir el modelo CRNN+CTC

La arquitectura sigue el diseño de Shi et al. (2015): un extractor convolucional
produce un mapa de características que se interpreta como una secuencia temporal,
alimentada a una BiLSTM cuya salida pasa por la pérdida CTC.

In [ ]:
modelo = construir_modelo_entrenamiento()
modelo.summary(line_length=100)

## 5. Entrenamiento (demo corta)

En un proyecto real entrenaríamos ~30 épocas con 50k muestras. Aquí mostramos sólo
2 épocas para ilustrar la dinámica de la pérdida.

In [ ]:
def gen(X, Y, bs=32):
    n = len(X)
    while True:
        idx = np.random.permutation(n)
        for i in range(0, n - bs + 1, bs):
            sel = idx[i:i+bs]
            yield {'imagen': X[sel], 'etiqueta': Y[sel].astype('float32')}, np.zeros(bs)

historia = modelo.fit(gen(X, Y), steps_per_epoch=20, epochs=2, verbose=1)

## 6. Inferencia y decodificación greedy

In [ ]:
inf = construir_modelo_inferencia()
inf.set_weights([w for w in modelo.get_weights() if w.shape != ()])

for i in range(4):
    pred = inf.predict(X[i:i+1], verbose=0)
    real = ''.join(ALFABETO[k] for k in Y[i] if k >= 0)
    print(f'Real: \"{real}\"   Predicho: \"{decodificar_greedy(pred)}\"')

## 7. Próximos pasos

- Entrenar más épocas (10–30) y aumentar el dataset.
- Sustituir EMNIST por IAM o Synth90k para palabras reales.
- Sustituir BiLSTM por Transformer (TrOCR) para mayor precisión.
- Integrar el modelo con la app Streamlit (`app/streamlit_app.py`).